# NYC Airbnb Exploratory Data Analysis
This notebook retrieves the versioned raw-data artifact from W&B, reviews data quality, and validates the stakeholder-approved cleaning rules before they are transferred into the reusable MLflow step.

In [ ]:
import pandas as pd
import wandb
from ydata_profiling import ProfileReport

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = run.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)
df.head()

## Initial quality profile
The profile is used to identify missing values, data types, duplicates, and extreme values. In particular, `last_review` initially arrives as text and `price` contains zero and unusually high values. Missing optional fields are intentionally not imputed here because the inference pipeline must handle them at prediction time.

In [ ]:
profile = ProfileReport(df, title="NYC Airbnb raw sample", minimal=True)
profile.to_notebook_iframe()

## Apply agreed cleaning rules
Stakeholders define a plausible nightly-price interval of $10–$350. We also remove exact duplicate records and rows without a price, convert review dates to datetime, and retain only listings inside the expected NYC latitude/longitude box used by the automated data checks.

In [ ]:
min_price, max_price = 10, 350
df = df.drop_duplicates().dropna(subset=["price"])
df = df[df["price"].between(min_price, max_price)].copy()
nyc_mask = (
    df["longitude"].between(-74.25, -73.50)
    & df["latitude"].between(40.5, 41.2)
)
df = df[nyc_mask].copy()
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

## Verification
The assertions below turn the EDA conclusions into executable checks. Missing values may remain in optional columns, as expected.

In [ ]:
assert df["price"].between(min_price, max_price).all()
assert df["longitude"].between(-74.25, -73.50).all()
assert df["latitude"].between(40.5, 41.2).all()
assert pd.api.types.is_datetime64_any_dtype(df["last_review"])
df.info()

## Close the tracked analysis run
Finishing explicitly flushes notebook metadata and logs to W&B.

In [ ]:
run.finish()